In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai


### Read sampled dataset with Amazon inventory data

In [2]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [3]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Camera & Photo,80x100 Monocular-Telescope Low Night Vision Mo...,3.6,187,[],[],29.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '80X100 monocular telescope high po...,MANDDDWU,"[Electronics, Camera & Photo, Binoculars & Sco...",{'Product Dimensions': '3.94 x 3.94 x 3.94 inc...,B09YHBXZC8,NaN,NaN,NaN
1,All Electronics,"SoundPEATS Air Conduction Headphones, RunFree ...",4.0,132,[𝗨𝗻𝗯𝗲𝗮𝘁𝗮𝗯𝗹𝗲 𝗖𝗼𝗺𝗳𝗼𝗿𝘁 𝗮𝗻𝗱 𝗟𝗶𝗴𝗵𝘁𝘄𝗲𝗶𝗴𝗵𝘁 𝗗𝗲𝘀𝗶𝗴𝗻 - 🍀...,[],29.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Soundpeats RunFree Lite - The Best...,SoundPEATS,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.48 x 5.16 x 2.13 inc...,B0BRXDR9GF,NaN,NaN,NaN
2,Computers,"𝟮𝟬𝟮𝟯 𝗟𝐚𝐭𝐞𝐬𝐭 Tablet 10.1"" Octa-Core Android 11 ...",4.3,411,[【VALUE FOR MONEY】Get all the bundled accessor...,[],179.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'The most complete tablet bundle', ...",FEONAL,"[Electronics, Computers & Accessories, Compute...","{'Standing screen display size': '10.1', 'Scre...",B0B4C98K37,NaN,NaN,NaN
3,Tools & Home Improvement,"120PCS 6 Inch Reusable Cable Ties, Wire Ties, ...",4.8,647,[Super Valuable: 120PCS cable straps is enough...,[],8.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': '120PCS Black Reusable Cable Ties',...",Mvyc,"[Electronics, Accessories & Supplies, Cord Man...","{'Manufacturer': 'Mvyc', 'Part Number': 'CAKF2...",B0B564JMB5,NaN,NaN,NaN
4,All Electronics,"CHIFENCHY 2 Packs Golf Speaker with Magnetic, ...",4.2,144,[GOLF PORTABLE BLUETOOTH SPEAKER: Designed in ...,[],99.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Honest Review of Ampcaddy Golf Blu...,CHIFENCHY,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '3.74 x 1.57 x 3.54 inc...,B0BDYW4FC4,NaN,NaN,NaN


In [8]:
list(df_items["features"].items())[1]

(1,
 ['𝗨𝗻𝗯𝗲𝗮𝘁𝗮𝗯𝗹𝗲 𝗖𝗼𝗺𝗳𝗼𝗿𝘁 𝗮𝗻𝗱 𝗟𝗶𝗴𝗵𝘁𝘄𝗲𝗶𝗴𝗵𝘁 𝗗𝗲𝘀𝗶𝗴𝗻 - 🍀 𝘽𝙚𝙨𝙩 𝙑𝙖𝙡𝙪𝙚 𝙊𝙥𝙚𝙣-𝙚𝙖𝙧 𝙎𝙥𝙤𝙧𝙩 𝙃𝙚𝙖𝙙𝙥𝙝𝙤𝙣𝙚 - 𝘾𝙉𝙀𝙏 🍀. The RunFree Lite headset is carefully optimized for weight distribution, ensuring a secure and comfortable fit that eliminates worries of slipping and shaking. Wrapped in skin-friendly liquid silicone, these lightweight headphones (0.99oz) provide a weightless and comfortable experience for extended listening during gym sessions, exercise, running, and more.',
  '𝗢𝗽𝗲𝗻-𝗘𝗮𝗿 𝗗𝗲𝘀𝗶𝗴𝗻 𝗮𝗻𝗱 𝗗𝘆𝗻𝗮𝗺𝗶𝗰 𝗦𝗼𝘂𝗻𝗱 - Stay connected and aware of your surroundings with our open-ear design sports headphones, which use air conduction technology to enhance the aerodynamic transmission of sound, resulting in a more dynamic and powerful audio experience. Enjoy up to 17 hours of playtime on a single charge to keep your music going for daily and weekly exercises. Only 1-2 hours charge will get full energy back.',
  '𝗣𝗼𝘄𝗲𝗿𝗳𝘂𝗹 𝗕𝗮𝘀𝘀 𝗮𝗻𝗱 𝗦𝘁𝗮𝗯𝗹𝗲 𝗖𝗼𝗻𝗻𝗲𝗰𝘁𝗶𝗼𝗻 - Featuring the latest low-frequency enhancement technology and B

In [9]:
list(df_items["images"].items())[1]

(1,
 [{'thumb': 'https://m.media-amazon.com/images/I/31OGeYLIfnL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/31OGeYLIfnL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/51IIaSHi-JL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41UxJVxZl2L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41UxJVxZl2L._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71y+++taYRL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51yay41lk+L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51yay41lk+L._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/71K+bE7AN0L._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/410gEVmQuAL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/410gEVmQuAL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/61+OhVOzGxL._AC_SL1

### Preprocess title and features

In [10]:
def preprocess_desciption(row):
        return f"{row['title']} {' '.join(row['features'])}"


def extract_first_large_image(row):
        return row['images'][0].get('large', '')


df_items["description"] = df_items.apply(preprocess_desciption, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [14]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,Camera & Photo,80x100 Monocular-Telescope Low Night Vision Mo...,3.6,187,[],80x100 Monocular-Telescope Low Night Vision Mo...,29.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '80X100 monocular telescope high po...,MANDDDWU,"[Electronics, Camera & Photo, Binoculars & Sco...",{'Product Dimensions': '3.94 x 3.94 x 3.94 inc...,B09YHBXZC8,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51Ni+NLJS4...
1,All Electronics,"SoundPEATS Air Conduction Headphones, RunFree ...",4.0,132,[𝗨𝗻𝗯𝗲𝗮𝘁𝗮𝗯𝗹𝗲 𝗖𝗼𝗺𝗳𝗼𝗿𝘁 𝗮𝗻𝗱 𝗟𝗶𝗴𝗵𝘁𝘄𝗲𝗶𝗴𝗵𝘁 𝗗𝗲𝘀𝗶𝗴𝗻 - 🍀...,"SoundPEATS Air Conduction Headphones, RunFree ...",29.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Soundpeats RunFree Lite - The Best...,SoundPEATS,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.48 x 5.16 x 2.13 inc...,B0BRXDR9GF,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31OGeYLIfn...
2,Computers,"𝟮𝟬𝟮𝟯 𝗟𝐚𝐭𝐞𝐬𝐭 Tablet 10.1"" Octa-Core Android 11 ...",4.3,411,[【VALUE FOR MONEY】Get all the bundled accessor...,"𝟮𝟬𝟮𝟯 𝗟𝐚𝐭𝐞𝐬𝐭 Tablet 10.1"" Octa-Core Android 11 ...",179.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'The most complete tablet bundle', ...",FEONAL,"[Electronics, Computers & Accessories, Compute...","{'Standing screen display size': '10.1', 'Scre...",B0B4C98K37,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51HfEQ1Osc...
3,Tools & Home Improvement,"120PCS 6 Inch Reusable Cable Ties, Wire Ties, ...",4.8,647,[Super Valuable: 120PCS cable straps is enough...,"120PCS 6 Inch Reusable Cable Ties, Wire Ties, ...",8.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': '120PCS Black Reusable Cable Ties',...",Mvyc,"[Electronics, Accessories & Supplies, Cord Man...","{'Manufacturer': 'Mvyc', 'Part Number': 'CAKF2...",B0B564JMB5,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51+jacwzqi...
4,All Electronics,"CHIFENCHY 2 Packs Golf Speaker with Magnetic, ...",4.2,144,[GOLF PORTABLE BLUETOOTH SPEAKER: Designed in ...,"CHIFENCHY 2 Packs Golf Speaker with Magnetic, ...",99.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Honest Review of Ampcaddy Golf Blu...,CHIFENCHY,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '3.74 x 1.57 x 3.54 inc...,B0BDYW4FC4,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51+uS0y5fA...


In [21]:
len(list(df_items['description'].items()))

1000

### Sample 50 items from this dataset

In [22]:
df_simple = df_items.sample(50, random_state=42)

In [23]:
len(df_simple)

50

In [37]:
data_to_embed = df_simple[['description', 'image', 'rating_number', 'price', 'average_rating', 'parent_asin']].to_dict(orient="records")

In [52]:
data_to_embed

[{'description': "USB C Hub, MCY USB C to HDMI Multiptort Adapter, 10 in 1 Portable Type C Dongle with 4K HDMI, VGA, PD Charging, USB 3.0 and USB 2.0 Ports, SD/TF Card, Compatible with MacBook, HP, Dell and More 【10 IN 1 USB C Docking Station】MCY Type-C smart docking station can solve 10 kinds of office troubles at one time. USB C hub multiport adapter hub rich interface. HDMI makes the big screen clearer. 1G files are transferred in 3 seconds, and there is no need to wait for the transfer. 【Wonderful more than one side】10-port USB C to HDMI adapter turns your laptop into a thin and light desktop computer. It is equipped with 4K@60Hz HDMI port, VGA port, SD/TF card reader and USB 3.0 and USB 2.0 ports. Get rid of the problem of insufficient notebook ports and connect more devices. A Type C interface helps you upgrade your space. 【4K Video HDMI to USB C Hub】Mirror or extend your laptop/tablet/phone's screen with hdmi port and directly stream 4K@60Hz (3840 x 2160) or full HD 1080P lifeli

### Define embedding function

In [46]:
from dotenv import load_dotenv
import os
import voyageai

load_dotenv()
VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

result = vo.embed(
    ["random data"],
    model="voyage-3",
    input_type="document"
)
embedding = result.embeddings[0]

In [47]:
embedding

[0.022841699421405792,
 -0.05015590041875839,
 0.00798660796135664,
 0.0023460660595446825,
 0.022681966423988342,
 0.06453179568052292,
 0.02779339626431465,
 0.00035440572537481785,
 0.005830223672091961,
 0.06325393915176392,
 -0.046002861112356186,
 0.014695358462631702,
 -0.06868483126163483,
 -0.008705402724444866,
 0.023800091817975044,
 0.033863216638565063,
 -0.057184115052223206,
 0.034981343895196915,
 0.07092107832431793,
 0.013896698132157326,
 -0.020126251503825188,
 0.030029647052288055,
 -0.04344714805483818,
 0.029550449922680855,
 -0.029710182920098305,
 0.017730269581079483,
 0.04823911190032959,
 0.010382590815424919,
 -0.013337635435163975,
 -0.0019367524655535817,
 -0.010542322881519794,
 -0.07124054431915283,
 -0.04440554231405258,
 -0.03881491348147392,
 0.005031562875956297,
 -0.03290482610464096,
 0.0027753463946282864,
 -0.03146723657846451,
 -0.02715446799993515,
 -0.06772643327713013,
 -0.025557145476341248,
 -0.024439020082354546,
 -0.051753219217061996,
 

In [44]:
len(embedding)

1024

In [49]:
def get_embedding(text, model = 'voyage-3'):
        result = vo.embed(
                [text],
                model=model,
                input_type="document"
        )
        return result.embeddings[0]

In [55]:
print(get_embedding("hi"))
len(get_embedding("hi")) #1024

[-0.015087280422449112, -0.031107794493436813, -0.043239835649728775, 0.008204680867493153, 0.002459459938108921, 0.05350540578365326, 0.0055994028225541115, -0.016331592574715614, -0.003266318468376994, 0.054127562791109085, 0.01889798603951931, -0.012754195369780064, -0.018586907535791397, -0.004763381090015173, -0.010965497232973576, 0.0012054269900545478, -0.006843714974820614, 0.051016781479120255, 0.06968145817518234, 0.00937122292816639, -0.020220065489411354, 0.02084222249686718, -0.004938362166285515, 0.003363530384376645, -0.06688176095485687, 0.01143211405724287, 0.0396624393761158, 0.006338213104754686, -0.0018956311978399754, 0.006415982730686665, -0.030330099165439606, -0.0650152936577797, -0.01882021501660347, -0.02286422811448574, -0.021308839321136475, 0.028463631868362427, 0.0035190691705793142, -0.03639611974358559, -0.0009332338231615722, -0.04852816089987755, 0.001817861688323319, -0.01796475052833557, -0.06563744693994522, 0.016720440238714218, -0.0563051067292690

1024

# Create qdrant collection

In [56]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant_client = QdrantClient(url = "http://localhost:6333")

# size is the size of vector from the embeding
qdrant_client.create_collection(
        collection_name="Amazon-items-collection-00",
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
) 

True

### Embed Data

In [57]:
# test
pointstruct = PointStruct(
        id = 0,
        vector = get_embedding("Test text"),
        payload={
                "text": "Test text",
                "model": "voyage-3",
        },
)
pointstruct

PointStruct(id=0, vector=[-0.0195973701775074, -0.04089885950088501, 0.015337072312831879, -0.003098398447036743, 0.02881510555744171, 0.019519910216331482, 0.03547666221857071, -0.021533869206905365, -1.4523742720484734e-05, 0.06196796894073486, -0.0005397991044446826, -0.013555493205785751, -0.05081373453140259, -0.013013273477554321, -0.023083068430423737, 0.014407552778720856, -0.008636785671114922, 0.02509702742099762, 0.03547666221857071, 0.026956066489219666, -0.011541534215211868, 0.02773066610097885, -0.01859039068222046, -0.006235526874661446, -0.050503894686698914, 0.04306773841381073, 0.020449429750442505, -0.012625973671674728, -0.01742849126458168, 0.001195788150653243, -0.05174325406551361, -0.07622060179710388, -0.02230846881866455, -0.024787187576293945, -0.010302174836397171, 0.021069109439849854, 0.0013458668254315853, -0.005925687029957771, -0.0063517168164253235, -0.03144874423742294, -0.011696454137563705, -0.040589019656181335, -0.03532174229621887, 0.04585629701

In [59]:
# Amazon data
pointstructs = []
for i, data in enumerate(data_to_embed):
        embedding_vector = get_embedding(data['description'])
        pointstructs.append(
                PointStruct(
                        id = i,
                        vector=embedding_vector,
                        payload=data
                )
        )

In [ ]:
pointstructs

[PointStruct(id=0, vector=[0.011272362433373928, -0.05210698023438454, -0.025216033682227135, 0.016497185453772545, -0.04183842986822128, 0.018875135108828545, -0.002366486005485058, 0.023298516869544983, -0.0069670965895056725, 0.029055040329694748, 0.004387729801237583, -0.017302565276622772, -0.012977196834981441, -0.026297006756067276, 0.036894794553518295, 0.011561647988855839, 0.05056579038500786, -0.06331108510494232, 0.02936510369181633, -0.045794349163770676, -0.019177470356225967, -0.009072755463421345, 0.015938643366098404, -0.05210696905851364, -0.011950409971177578, -0.004672300536185503, 0.05598358437418938, -0.006347178481519222, -0.03240886330604553, 0.03264899551868439, -0.051971204578876495, -0.01515740342438221, 0.04154471307992935, -0.03105265460908413, -0.015630856156349182, 0.045289114117622375, -0.016751300543546677, 0.03467434644699097, -0.03734800964593887, 0.003195543307811022, -0.020759038627147675, 0.01831577531993389, -0.022193865850567818, 0.03236408159136

In [61]:
len(pointstructs)

50

### Write embedding to qdrant

In [62]:
qdrant_client.upsert(
        collection_name = 'Amazon-items-collection-00',
        wait = True,
        points = pointstructs,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Define retrieval from qdrant

In [65]:
def retrieve_data(query, k=5):
        """
        Get top k nearest data points from qdrant
        """
        query_embedding = get_embedding(query)
        results = qdrant_client.query_points(
                collection_name='Amazon-items-collection-00',
                query=query_embedding,
                limit=k
        )
        return results

In [66]:
retrieve_data("What kind of charging cords do you offer?", k=10).points

[ScoredPoint(id=38, version=1, score=0.54527074, payload={'description': '2022 Updated Digital TV Antenna to 500 Miles Range ', 'image': 'https://m.media-amazon.com/images/I/41sZZKxw3CL._AC_.jpg', 'rating_number': 131, 'price': None, 'average_rating': 3.4, 'parent_asin': 'B0B4WNFTVZ'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=20, version=1, score=0.52906954, payload={'description': 'Wireless Earbud, Bluetooth 5.1 Headphones with Microphone Deep Bass Bluetooth Earphones in-Ear, CVC8.0 Noise Cancelling Earbud for Sport Running Gym IPX7 Waterproof, 30H Playtime, Touch Control White ', 'image': 'https://m.media-amazon.com/images/I/41AmQP5zJ0L._AC_.jpg', 'rating_number': 457, 'price': None, 'average_rating': 4.1, 'parent_asin': 'B09F36P17Y'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=7, version=1, score=0.45518002, payload={'description': "iPhone Fast Charger, [Apple MFi Certified] 20W PD Type C Power Wall Charger with 3FT Lightning Cable Compati